In [2]:
# Cell — So sánh metrics trên Test cho Decision Tree và XGBoost
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Paths (thay đổi nếu bạn đặt khác)
DT_DIR = Path("./checkpoint/dtree")
XGB_DIR = Path("./checkpoint/xgb")

DT_MODEL_PKL = DT_DIR / "dtree_model.pkl"        # pipeline đã lưu (imputer + tree)
DT_PRED_CSV   = DT_DIR / "predictions_test.csv"  # nếu đã lưu

XGB_MODEL_JSON = XGB_DIR / "xgb_model.json"
XGB_MODEL_PKL  = XGB_DIR / "hgb_model.pkl"       # fallback if HGB used
XGB_PRED_CSV   = XGB_DIR / "predictions_test.csv"

OUT_METRICS_CSV = Path("./checkpoint/metrics_comparison.csv")

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def summarize(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    abs_err = np.abs(y_true - y_pred)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred, squared=False)),
        "R2": float(r2_score(y_true, y_pred)),
        "N": int(len(y_true))
    }

results = {}

# Try to use X_test/y_test from scope if present
gl = globals()
have_Xy = ("X_test" in gl and "y_test" in gl)
if have_Xy:
    X_test_local = gl["X_test"]
    y_test_local = gl["y_test"]
else:
    X_test_local = None
    y_test_local = None

# 1) Decision Tree / pipeline
dt_metrics = None
dt_pred_df = None
try:
    # If pipeline exists and X_test in scope, use it to predict
    if DT_MODEL_PKL.exists() and X_test_local is not None:
        import joblib
        pipe = joblib.load(DT_MODEL_PKL)
        yhat_dt = pipe.predict(X_test_local)
        ytrue_dt = np.array(y_test_local)
        dt_metrics = summarize(ytrue_dt, yhat_dt)
    # else try to read saved predictions csv
    elif DT_PRED_CSV.exists():
        dt_pred_df = pd.read_csv(DT_PRED_CSV, parse_dates=["thoi_diem"])
        ytrue_dt = dt_pred_df["y_true"].values
        yhat_dt  = dt_pred_df["y_pred"].values
        dt_metrics = summarize(ytrue_dt, yhat_dt)
    else:
        dt_metrics = None
        print("[WARN] Không tìm thấy pipeline DecisionTree hoặc predictions CSV; bỏ qua Decision Tree.")
except Exception as e:
    dt_metrics = None
    print("[ERROR] Khi đánh giá Decision Tree:", e)

if dt_metrics is not None:
    results["DecisionTree"] = dt_metrics

# 2) XGBoost (or HGB)
xgb_metrics = None
xgb_pred_df = None
try:
    # If X_test in scope and model file exists try to load model and predict
    if X_test_local is not None and XGB_MODEL_JSON.exists():
        # XGBoost model saved as JSON
        try:
            from xgboost import XGBRegressor
            model_xgb = XGBRegressor()
            model_xgb.load_model(str(XGB_MODEL_JSON))
            yhat_xgb = model_xgb.predict(X_test_local)
            ytrue_xgb = np.array(y_test_local)
            xgb_metrics = summarize(ytrue_xgb, yhat_xgb)
        except Exception as e:
            # fallback try joblib pkl
            if XGB_MODEL_PKL.exists():
                import joblib
                model_x = joblib.load(XGB_MODEL_PKL)
                yhat_xgb = model_x.predict(X_test_local)
                ytrue_xgb = np.array(y_test_local)
                xgb_metrics = summarize(ytrue_xgb, yhat_xgb)
            else:
                raise e
    elif XGB_PRED_CSV.exists():
        xgb_pred_df = pd.read_csv(XGB_PRED_CSV, parse_dates=["thoi_diem"])
        ytrue_xgb = xgb_pred_df["y_true"].values
        yhat_xgb  = xgb_pred_df["y_pred"].values
        xgb_metrics = summarize(ytrue_xgb, yhat_xgb)
    else:
        # try load XGB_MODEL_JSON even if X_test not present -> but we need X_test to predict
        xgb_metrics = None
        print("[WARN] Không có X_test trong scope và không tìm thấy predictions XGB CSV; bỏ qua XGBoost.")
except Exception as e:
    xgb_metrics = None
    print("[ERROR] Khi đánh giá XGBoost/HGB:", e)

if xgb_metrics is not None:
    results["XGBoost"] = xgb_metrics

# 3) Nếu cả hai có preds, tạo DataFrame so sánh
if not results:
    print("Không có kết quả để báo cáo. Hãy đảm bảo bạn đã chạy train và/hoặc đã lưu predictions_test.csv cho mỗi model.")
else:
    df_out = pd.DataFrame(results).T
    # sắp cột
    cols_order = ["N","MAE","RMSE","MED_ABS_ERR","P90_ABS_ERR","R2","MAPE_%"]
    df_out = df_out[[c for c in cols_order if c in df_out.columns]]
    # in ra đẹp
    pd.options.display.float_format = "{:,.4f}".format
    print("\n=== So sánh metrics trên Test set ===")
    print(df_out)
    # Lưu kết quả
    OUT_METRICS_CSV.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(OUT_METRICS_CSV)
    print(f"\nĐã lưu kết quả so sánh vào: {OUT_METRICS_CSV}")



=== So sánh metrics trên Test set ===
                      N    MAE   RMSE     R2
DecisionTree 6,653.0000 1.3386 3.2392 0.8649
XGBoost      6,653.0000 1.5412 3.5458 0.8381

Đã lưu kết quả so sánh vào: checkpoint/metrics_comparison.csv


In [4]:
# Cell — Evaluate CNN (pth) + DecisionTree + XGBoost on Test set (MAE, RMSE, R2)
import os, json, glob
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Paths (thực thi trong repo của bạn)
OUT_DIR_DT   = Path("./checkpoint/dtree")
OUT_DIR_XGB  = Path("./checkpoint/xgb")
OUT_DIR_CNN  = Path("./checkpoint/cnn_lstm")
CNN_PTH      = Path("./checkpoint/cnn_lstm/cnn_lstm_model_seq6_28d.pth")   # bạn upload .pth
CNN_PRED_CSV = Path("./checkpoint/cnn_lstm/predictions_cnn_lstm_seq6_28d.csv")
DT_PRED_CSV  = OUT_DIR_DT / "predictions_test.csv"
XGB_PRED_CSV = OUT_DIR_XGB / "predictions_test.csv"
MET_OUT = Path("./checkpoint/metrics_comparison.csv")

# utility
def mape(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask]-y_pred[mask]) / y_true[mask]))*100

def summarize(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return {
        "N": int(len(y_true)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred, squared=False)),
        "R2": float(r2_score(y_true, y_pred)),
        "MAPE_%": float(mape(y_true, y_pred))
    }

results = {}
saved_preds = {}

# 1) Try to load DecisionTree predictions CSV
if DT_PRED_CSV.exists():
    df = pd.read_csv(DT_PRED_CSV, parse_dates=["thoi_diem"])
    if {"y_true","y_pred"}.issubset(df.columns):
        metrics = summarize(df["y_true"].values, df["y_pred"].values)
        results["DecisionTree"] = metrics
        saved_preds["DecisionTree"] = df.copy()
        print("[OK] Loaded DecisionTree preds from", DT_PRED_CSV)
    else:
        print("[WARN] DT predictions CSV exists but missing columns y_true/y_pred:", DT_PRED_CSV)

# 2) Try to load XGBoost predictions CSV
if XGB_PRED_CSV.exists():
    df = pd.read_csv(XGB_PRED_CSV, parse_dates=["thoi_diem"])
    if {"y_true","y_pred"}.issubset(df.columns):
        metrics = summarize(df["y_true"].values, df["y_pred"].values)
        results["XGBoost"] = metrics
        saved_preds["XGBoost"] = df.copy()
        print("[OK] Loaded XGBoost preds from", XGB_PRED_CSV)
    else:
        print("[WARN] XGB predictions CSV exists but missing columns y_true/y_pred:", XGB_PRED_CSV)

# 3) Try to load CNN predictions CSV
if CNN_PRED_CSV.exists():
    df = pd.read_csv(CNN_PRED_CSV, parse_dates=["thoi_diem"])
    if {"y_true","y_pred"}.issubset(df.columns):
        metrics = summarize(df["y_true"].values, df["y_pred"].values)
        results["CNN_LSTM"] = metrics
        saved_preds["CNN_LSTM"] = df.copy()
        print("[OK] Loaded CNN preds from", CNN_PRED_CSV)
    else:
        print("[WARN] CNN predictions CSV exists but missing columns y_true/y_pred:", CNN_PRED_CSV)

# 4) If any model missing, try to compute using model files + X_test/y_test if available in scope
gl = globals()
# helper to check X_test/y_test in scope
X_test_in_scope = "X_test" in gl and "y_test" in gl
if X_test_in_scope:
    X_test_scope = gl["X_test"]; y_test_scope = gl["y_test"]
else:
    X_test_scope = None; y_test_scope = None

# DecisionTree: if not in results, try load pipeline and predict using X_test if available
if "DecisionTree" not in results:
    try:
        dt_pickle = OUT_DIR_DT / "dtree_model.pkl"
        if dt_pickle.exists() and X_test_scope is not None:
            import joblib
            pipe = joblib.load(dt_pickle)
            yhat = pipe.predict(X_test_scope)
            metrics = summarize(y_test_scope, yhat)
            results["DecisionTree"] = metrics
            df = pd.DataFrame({"thoi_diem": getattr(X_test_scope, "index", np.arange(len(yhat))),
                               "y_true": y_test_scope, "y_pred": yhat})
            saved_preds["DecisionTree"] = df
            print("[OK] Predicted DecisionTree from pipeline and X_test in scope.")
        else:
            print("[WARN] Cannot evaluate DecisionTree: missing predictions CSV and missing pipeline or X_test.")
    except Exception as e:
        print("[ERROR] DecisionTree eval failed:", e)

# XGBoost: similar
if "XGBoost" not in results:
    try:
        xgb_json = OUT_DIR_XGB / "xgb_model.json"
        xgb_pkl  = OUT_DIR_XGB / "hgb_model.pkl"
        if X_test_scope is not None:
            if xgb_json.exists():
                try:
                    from xgboost import XGBRegressor
                    model_xgb = XGBRegressor()
                    model_xgb.load_model(str(xgb_json))
                    yhat = model_xgb.predict(X_test_scope)
                    metrics = summarize(y_test_scope, yhat)
                    results["XGBoost"] = metrics
                    df = pd.DataFrame({"thoi_diem": getattr(X_test_scope, "index", np.arange(len(yhat))),
                                       "y_true": y_test_scope, "y_pred": yhat})
                    saved_preds["XGBoost"] = df
                    print("[OK] Predicted XGBoost from JSON model and X_test in scope.")
                except Exception as e:
                    print("[WARN] XGBoost JSON load/predict failed:", e)
            elif xgb_pkl.exists():
                import joblib
                model_hgb = joblib.load(xgb_pkl)
                yhat = model_hgb.predict(X_test_scope)
                metrics = summarize(y_test_scope, yhat)
                results["XGBoost"] = metrics
                df = pd.DataFrame({"thoi_diem": getattr(X_test_scope, "index", np.arange(len(yhat))),
                                   "y_true": y_test_scope, "y_pred": yhat})
                saved_preds["XGBoost"] = df
                print("[OK] Predicted HGB from pkl and X_test in scope.")
            else:
                print("[WARN] No XGB model file found (json/pkl) or X_test not available.")
        else:
            print("[WARN] Missing X_test in scope; cannot predict XGBoost unless predictions CSV exists.")
    except Exception as e:
        print("[ERROR] XGBoost eval failed:", e)

# CNN: if not in results, try to build predictions from .pth and scalers (best-effort)
if "CNN_LSTM" not in results:
    try:
        # first check if we have sequences and scalers saved from original notebook in scope
        have_seq = "Xseq_te_s" in gl and "y_te" in gl and "y_scaler" in gl
        if have_seq:
            Xseq_te_s = gl["Xseq_te_s"]; y_te = gl["y_te"]
            model_path_candidates = [
                CNN_PTH,
                Path("./checkpoint/cnn_lstm/cnn_lstm_model_seq6_28d.pth")
            ]
            model_path = next((p for p in model_path_candidates if p.exists()), None)
            if model_path is None:
                print("[WARN] CNN .pth not found in common locations; cannot load model.")
            else:
                import torch, torch.nn as nn
                # define CNNLSTM class matching training architecture
                class CNNLSTM(nn.Module):
                    def __init__(self, seq_feat_dim, conv_filters=16, lstm_hidden=64, dense_hidden=32, dropout=0.1):
                        super().__init__()
                        self.conv = nn.Sequential(
                            nn.Conv1d(seq_feat_dim, conv_filters, kernel_size=5, padding=2),
                            nn.ReLU(),
                            nn.MaxPool1d(2),
                            nn.Conv1d(conv_filters, conv_filters, kernel_size=5, padding=2),
                            nn.ReLU(),
                            nn.MaxPool1d(2)
                        )
                        self.lstm = nn.LSTM(input_size=conv_filters, hidden_size=lstm_hidden, batch_first=True)
                        self.norm = nn.LayerNorm(lstm_hidden)
                        self.dropout = nn.Dropout(dropout)
                        self.head = nn.Sequential(
                            nn.Linear(lstm_hidden, dense_hidden), nn.ReLU(),
                            nn.Linear(dense_hidden, 1)
                        )
                    def forward(self, x_seq):
                        x = x_seq.transpose(1, 2)
                        x = self.conv(x)
                        x = x.transpose(1, 2)
                        out, _ = self.lstm(x)
                        h = out[:, -1, :]
                        h = self.norm(h)
                        h = self.dropout(h)
                        return self.head(h).squeeze(1)

                device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                # instantiate with feature dim from Xseq_te_s
                seq_feat_dim = int(Xseq_te_s.shape[2])
                model = CNNLSTM(seq_feat_dim=seq_feat_dim).to(device)
                model.load_state_dict(torch.load(model_path, map_location=device))
                model.eval()
                # predict in batches
                import math, torch
                BS = 256
                preds = []
                with torch.no_grad():
                    for i in range(0, len(Xseq_te_s), BS):
                        xb = torch.tensor(Xseq_te_s[i:i+BS], dtype=torch.float32, device=device)
                        yhat = model(xb).detach().cpu().numpy().ravel()
                        preds.append(yhat)
                yhat_s = np.concatenate(preds, axis=0)
                # inverse scale y if y_scaler in scope
                if "y_scaler" in gl:
                    y_scaler = gl["y_scaler"]
                    try:
                        y_pred = y_scaler.inverse_transform(yhat_s.reshape(-1,1)).ravel()
                    except Exception:
                        y_pred = yhat_s  # fallback
                else:
                    y_pred = yhat_s
                y_true = y_te
                metrics = summarize(y_true, y_pred)
                results["CNN_LSTM"] = metrics
                df = pd.DataFrame({"thoi_diem": getattr(y_te, "index", np.arange(len(y_true))),
                                   "y_true": y_true, "y_pred": y_pred})
                saved_preds["CNN_LSTM"] = df
                print("[OK] Predicted CNN using Xseq_te_s and .pth (scalers used if available in scope).")
        else:
            # fallback: try to load existing predictions CSVs already attempted earlier (none), so we cannot proceed
            print("[WARN] Cannot reconstruct CNN predictions: Xseq_te_s/y_te not in scope and predictions CSV missing.")
            # Attempt to find predictions CSV under other common paths
            alt = list(Path("./").rglob("predictions_cnn*.csv"))
            if alt:
                p = alt[0]
                df = pd.read_csv(p, parse_dates=["thoi_diem"])
                if {"y_true","y_pred"}.issubset(df.columns):
                    results["CNN_LSTM"] = summarize(df["y_true"].values, df["y_pred"].values)
                    saved_preds["CNN_LSTM"] = df.copy()
                    print("[OK] Found alternate CNN preds CSV:", p)
                else:
                    print("[WARN] Alternate CNN preds CSV found but missing y_true/y_pred:", p)
            else:
                print("[WARN] No alternate CNN predictions CSV found in repo.")
    except Exception as e:
        print("[ERROR] CNN eval attempt failed:", e)

# 5) Summarize & save metrics table
if not results:
    print("Không có metrics nào được tính. Kiểm tra lại các file predictions/model/X_test trong notebook.")
else:
    df_out = pd.DataFrame(results).T
    # reorder columns
    cols = ["N","MAE","RMSE","R2"]
    present_cols = [c for c in cols if c in df_out.columns]
    df_out = df_out[present_cols]
    pd.options.display.float_format = "{:,.4f}".format
    print("\n=== Metrics summary ===")
    print(df_out)
    # save
    MET_OUT.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(MET_OUT)
    print("\nSaved metrics comparison to:", MET_OUT)

    # Also save any reconstructed preds to checkpoint for later plotting
    for name, pdf in saved_preds.items():
        p = Path("./checkpoint") / f"predictions_{name}.csv"
        pdf.to_csv(p, index=False)
        print("Saved predictions for", name, "->", p)


[OK] Loaded DecisionTree preds from checkpoint/dtree/predictions_test.csv
[OK] Loaded XGBoost preds from checkpoint/xgb/predictions_test.csv
[OK] Loaded CNN preds from checkpoint/cnn_lstm/predictions_cnn_lstm_seq6_28d.csv

=== Metrics summary ===
                      N    MAE   RMSE     R2
DecisionTree 6,653.0000 1.3386 3.2392 0.8649
XGBoost      6,653.0000 1.5412 3.5458 0.8381
CNN_LSTM     6,653.0000 2.0146 3.6451 0.8289

Saved metrics comparison to: checkpoint/metrics_comparison.csv
Saved predictions for DecisionTree -> checkpoint/predictions_DecisionTree.csv
Saved predictions for XGBoost -> checkpoint/predictions_XGBoost.csv
Saved predictions for CNN_LSTM -> checkpoint/predictions_CNN_LSTM.csv


### Ý nghĩa các độ đo đánh giá mô hình hồi quy

#### 1️⃣ MAE — Mean Absolute Error (Sai số tuyệt đối trung bình)
- **Định nghĩa:** Trung bình của giá trị tuyệt đối $|y - \hat{y}|$.  
- **Diễn giải:** Thể hiện sai số điển hình giữa giá trị thực và giá trị dự đoán.  
- **Đơn vị:** Giống với đơn vị của biến đích (ở đây là mực nước).  
- **Đặc điểm:** Ít nhạy với outlier hơn RMSE; giá trị càng nhỏ càng tốt.

#### 2️⃣ RMSE — Root Mean Squared Error (Căn bậc hai sai số bình phương trung bình)
- **Định nghĩa:** $\sqrt{\text{mean}((y - \hat{y})^2)}$.  
- **Diễn giải:** Đo mức sai lệch trung bình, **phạt nặng hơn các sai số lớn**.  
- **Đơn vị:** Giống với đơn vị của biến đích.  
- **Đặc điểm:** Nhạy với các điểm sai số lớn (outlier hoặc đỉnh cực trị); giá trị càng nhỏ càng tốt.

#### 3️⃣ R² — Coefficient of Determination (Hệ số xác định)
- **Định nghĩa:** $R^2 = 1 - \frac{\text{SSE}}{\text{SST}}$, trong đó SSE là tổng bình phương sai số và SST là tổng bình phương độ lệch so với trung bình.  
- **Diễn giải:** Cho biết tỷ lệ biến thiên của dữ liệu thực được mô hình giải thích.  
- **Đơn vị:** Không có đơn vị.  
- **Đặc điểm:** 
  - $R^2 \approx 1$: mô hình giải thích tốt biến thiên dữ liệu.  
  - $R^2 \approx 0$: mô hình yếu, giải thích kém.  
  - $R^2 < 0$: mô hình tệ hơn cả việc dự đoán trung bình.  

#### 💡 Gợi ý sử dụng
- **MAE:** để mô tả sai số điển hình (phù hợp báo cáo vận hành).  
- **RMSE:** để kiểm soát sai số lớn, nhất là trong các sự kiện cực trị.  
- **R²:** để đánh giá khả năng tổng quát của mô hình trong việc giải thích dữ liệu.
